# 未见 Agent 模型来源的任务留出检验

**目的。** 检查命令特征在训练时完全排除一个 Agent 来源后，能否区分公开发布中的人类记录与该来源的 Agent 记录；同时判断高分是否只是低维命令数量/动作比例能复现。

**实验对象。** Cyber Task Horizons 固定版本中的 177 条非空公开 CTF 轨迹、40 个 benchmark/task 组：57 条专家终端记录，120 条 Agent 工具调用记录；模型来源为 Claude Opus 4.6、Claude Sonnet 4.6 和 GPT-5.5。任务组整体进行五折留出。每个测试折轮流将目标模型的全部 Agent 轨迹排除在训练集之外。模型名只决定留出对象，不是特征。报告等权基准宏平均以及按任务簇、基准分层的 5,000 次 bootstrap 区间。

**主要限制。** 人类标签与终端采集路径绑定，Agent 标签与结构化工具调用路径绑定（177 条记录无不匹配）。因此，本实验测得的是这份发布中两类记录标签的可分性和模型来源压力测试，不能确立通用 Agent 检测器，更不能说明 APT 活动关联效果。数据中的任务是 CTF 题目，不是攻击活动或组织标签。本分析是事后探索性检验，不是独立确认实验。

预处理、折分、特征和全部指标见 [`scripts/run_unseen_agent_model.py`](../scripts/run_unseen_agent_model.py)、[`results/unseen_agent_model_generalization.json`](../results/unseen_agent_model_generalization.json) 与任务级 CSV。

In [ ]:
from pathlib import Path
import csv, json, subprocess, sys
from IPython.display import display, Image

ROOT = Path.cwd()
if (ROOT / 'research' / 'scripts').is_dir(): ROOT = ROOT / 'research'
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts/run_unseen_agent_model.py').exists(), ROOT

# 设为 True 可用当前 Python 环境完整重跑模型、bootstrap 与图表。
RERUN = False
if RERUN:
    subprocess.run([sys.executable, str(ROOT / 'scripts/run_unseen_agent_model.py')], check=True, cwd=ROOT)

result = json.loads((ROOT / 'results/unseen_agent_model_generalization.json').read_text())
assert result['data']['traces'] == 177 and result['data']['matched_task_groups'] == 40
assert result['data']['capture_path_label_audit']['mismatches'] == 0
print('数据版本:', result['data']['dataset_revision'])
print('轨迹 / 任务:', result['data']['traces'], '/', result['data']['matched_task_groups'])
print('采集路径与标签不匹配:', result['data']['capture_path_label_audit']['mismatches'])
print('协议:', result['protocol']['outer_split'])


In [ ]:
# 命令 n-gram：按留出 Agent 来源和基准列出“全来源训练”与“排除该来源训练”的配对结果。
rows = [r for r in result['summary'] if r.get('feature') == 'command_tfidf']
columns = ('heldout_agent_model', 'benchmark', 'task_count',
           'all_models_balanced_accuracy', 'unseen_model_balanced_accuracy',
           'delta_balanced_accuracy')
for row in rows:
    if row.get('benchmark'):
        print(' | '.join(f"{row.get(k)}" if isinstance(row.get(k), str) else f"{row.get(k):.3f}" if isinstance(row.get(k), float) else str(row.get(k)) for k in columns))


In [ ]:
# 跨三个留出模型来源的整体汇总与不确定性区间。
for row in rows:
    if row.get('heldout_agent_model') == 'macro across three held-out models':
        for label, metric in [('All-source BAcc', 'all_models_balanced_accuracy'),
                              ('Unseen-source BAcc', 'unseen_model_balanced_accuracy'),
                              ('Paired delta', 'delta_balanced_accuracy'),
                              ('Unseen-source macro-F1', 'unseen_model_macro_f1'),
                              ('Unseen-source ROC-AUC', 'unseen_model_roc_auc')]:
            ci = row[f'{metric}_95_task_cluster_ci']
            print(f"{label}: {row[metric]:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]")


In [ ]:
# 展示实验脚本生成的蓝色论文图。
display(Image(filename=str(ROOT / 'results/figures/figureA6_unseen_model_generalization.png')))


## 结论边界

若命令 n-gram 的结果在排除一个 Agent 来源后仍接近全来源训练基线，说明可在这份发布中观察到跨模型来源迁移；它并不能排除终端与工具调用采集路径差异。判断模型分数是否能支持活动关联，仍需共同遥测、固定脚本对照和独立活动 ID，并直接比较误拆分和误合并。分类效果本身不构成关联效用证据。